In [57]:
from cgra import *
from kernels import *
from sat_to_csv import *

In [58]:
kernel_name = "mmul_os_opt2"
version = ""

In [59]:
# Global variables
CGRA_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr_A = 20000
#first_addr_B = first_addr_A + rowsA*colsA*4
#first_addr_C = first_addr_B + colsA*colsB*4
first_addr_B = 30000
first_addr_C = 40000
end_addr_C = first_addr_C

In [60]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [61]:
# Data
def configMemory(A_data, B_data, rowsA, colsA, colsB):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # &B[0][0]          &C[0][1]        &A[0][0]        nRowsBlocksC
    # nColsBlocksC      &B[0][1]        &C[1][2]        &A[1][0]
    # &A[2][0]          loopColsA       &B[0][2]        &C[2][3]
    # &C[3][0]          &A[3][0]        nColsBlocksC    &B[0][3]
    # ----------------------
    # -4*colsB          colsA           -               -
    # -                 -4*colsB        colsA           -
    # -                 -               -4*colsB        colsA
    # colsA             -               -               -4*colsB
    nItLoopColsA = colsA
    nColsBlocksC = int(colsB/CGRA_N_ROWS)
    nRowsBlocksC = int(rowsA/CGRA_N_ROWS)
    config_vals_col0 = [first_addr_B, nColsBlocksC, first_addr_A + 2*colsA*4, first_addr_C + 3*colsB*4, -4*colsB, colsA]
    config_vals_col1 = [first_addr_C + 4, first_addr_B + 4, nItLoopColsA, first_addr_A + 3*colsA*4, colsA, -4*colsB]
    config_vals_col2 = [first_addr_A, first_addr_C + 2*4 + colsB*4, first_addr_B + 2*4, nColsBlocksC, colsA, -4*colsB]
    config_vals_col3 = [nRowsBlocksC, first_addr_A + colsA*4, first_addr_C + 3*4 + 2*colsB*4, first_addr_B + 3*4, colsA, -4*colsB]
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [62]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [63]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [64]:
def mmul_cpu(A_data, B_data, rowsA, colsA, colsB):
    expected_res = [0 for _ in range(rowsA*colsB)]
    for rA in range(rowsA):
        for cB in range(colsB):
            sum = 0
            for cA in range(colsA):
                sum += A_data[rA*colsA + cA] * B_data[cA*colsB + cB]
            expected_res[rA*colsB + cB] = sum
    return expected_res

In [65]:
# Test dimensions (4xXx4)
rowsA = 8
colsA = 5
colsB = 9
A_data = list(range(0, rowsA * colsA))
B_data = [x + 100 for x in range(0, colsA * colsB)]
print("A")
printAsMatrix(A_data, rowsA, colsA)
print("B")
printAsMatrix(B_data, colsA, colsB)
load_addrs = configMemory(A_data, B_data, rowsA, colsA, colsB)

In [66]:
runKernel(load_addrs, max_it=1000*rowsA)

In [67]:
result = getResult(first_addr_C, first_addr_C + rowsA*colsB*4, rowsA, colsB)
# Process estra rows/cols
if rowsA%4 != 0:
    for rA in range(rowsA - rowsA%4, rowsA):
        for cB in range(colsB):
            for k in range(colsA):
                result[rA*colsB+cB] += A_data[rA*colsA+k]*B_data[k*colsB+cB]
if colsB%4 != 0:
    for cB in range(colsB - colsB%4, colsB):
        for rA in range(rowsA - rowsA%4):
            for k in range(colsA):
                result[rA*colsB+cB] += A_data[rA*colsA+k]*B_data[k*colsB+cB]


expected_res = mmul_cpu(A_data, B_data, rowsA, colsA, colsB)


if len(result) < len(expected_res):
    print(len(result))
    printAsMatrix(result, rowsA, colsB)
    print(len(expected_res))
    printAsMatrix(expected_res, rowsA, colsB)
else:
    errors = 0
    for i in range(len(expected_res)):
        if expected_res[i] != result[i]:
            errors += 1
    if errors > 0:
        print("Err: " + str(errors))
        print(len(result))
        printAsMatrix(result, rowsA, colsB)
        print(len(expected_res))
        printAsMatrix(expected_res, rowsA, colsB)
    else:
        print("OK")

